In [2]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv('apartments.csv', encoding='latin1').drop(22)

In [4]:
df.iloc[21]

PropertyName                               Krisumi Waterfall Residences
PropertySubName       1, 2, 3, 4 BHK Apartment, 1 RK Studio Apartmen...
NearbyLocations       ['Dwarka Expy', 'Delhi Public School, Sector 8...
LocationAdvantages    {'Dwarka Expy': '6 KM', 'Delhi Public School, ...
Link                  https://www.99acres.com/krisumi-waterfall-resi...
PriceDetails          {'1 BHK': {'building_type': 'Apartment', 'area...
TopFacilities         ['Concierge Service', 'Swimming Pool', 'Bar/Ch...
Name: 21, dtype: object

In [5]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."


In [6]:
df.iloc[1].NearbyLocations # This can be deleted because we have LocationAdvantages. NearbyLocations is a subset of LocationAdvantages.

"['DPSG Palam Vihar Gurugram', 'The NorthCap University', 'Park Hospital, Palam Vihar', 'Pacific D21 Mall', 'Palam Vihar Halt Railway Station']"

In [7]:
df.iloc[1].LocationAdvantages

"{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The NorthCap University': '4.4 Km', 'Park Hospital, Palam Vihar': '1.4 Km', 'Pacific D21 Mall': '8.2 Km', 'Palam Vihar Halt Railway Station': '1.2 Km', 'Dwarka Sector 21 Metro Station': '8.1 Km', 'Dwarka Expressway': '450 m', 'Fun N Food Water Park': '8.1 Km', 'Indira Gandhi International Airport': '14.1 Km', 'Tau DeviLal Sports Complex': '11.2 Km', 'Hamoni Golf Camp': '5 Km', 'Hyatt Place': '6.1 Km', 'Altrade Business Centre': '11.2 Km'}"

In [8]:
df.iloc[6].PriceDetails

"{'3 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,015 - 2,150 sq.ft.', 'price-range': '? 1.53 - 1.85 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,250 - 2,675 sq.ft.', 'price-range': '? 1.71 - 2.46 Cr'}}"

In [9]:
df.iloc[1].TopFacilities

"['Bowling Alley', 'Mini Theatre', 'Manicured Garden', 'Swimming Pool', 'Flower Garden', 'Reading Lounge', 'Golf Course', 'Barbecue', 'Sauna']"

In [10]:
df[['PropertyName', 'TopFacilities']]['TopFacilities'][0]

"['Swimming Pool', 'Salon', 'Restaurant', 'Spa', 'Cafeteria', 'Sun Deck', '24x7 Security', 'Club House', 'Gated Community']"

In [11]:
def extract_list(s):
    return re.findall(r"'(.*?)'", s)
df['TopFacilities'] = df['TopFacilities'].apply(extract_list)

In [12]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Salon, Restaurant, Spa, Cafete..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","[Bowling Alley, Mini Theatre, Manicured Garden..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"[Terrace Garden, Gazebo, Fountain, Amphitheatr..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Volley Ball Court, Aerobics Ce..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"[Mini Theatre, Doctor on Call, Concierge Servi..."


In [13]:
df['FacilitiesStr'] = df['TopFacilities'].apply(' '.join)

In [14]:
df['FacilitiesStr'][0]

'Swimming Pool Salon Restaurant Spa Cafeteria Sun Deck 24x7 Security Club House Gated Community'

In [15]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [16]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df['FacilitiesStr'])

In [17]:
tfidf_matrix.toarray()[0]

array([0.        , 0.        , 0.        , 0.18809342, 0.18809342,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [18]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [19]:
cosine_sim1.shape

(246, 246)

In [20]:
df[['PropertyName', 'PriceDetails']]['PriceDetails'][1]

"{'3 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '1,605 - 2,170 sq.ft.', 'price-range': '? 2.2 - 3.03 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,248 - 2,670 sq.ft.', 'price-range': '? 3.08 - 3.73 Cr'}}"

In [21]:
def recommend_properties(property_name, cosine_sim=cosine_sim1):
    # Get the index of the property that matches the name
    idx = df.index[df['PropertyName'] == property_name].to_list()[0]
    
    # Get the pairwise similarity scores of all properties with that property
    sim_scores = list(enumerate(cosine_sim1[idx]))

    # Sort the properties based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)   

    # Get the scores of the 5 most similar properties
    sim_scores = sim_scores[1:6]
    
    # Get the property indices
    property_indices = [i[0] for i in sim_scores]
    
    recommendations_df = pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[property_indices],
        'SimilarityScore': sim_scores
    })

    # return the top 10 most similar properties
    return recommendations_df

In [22]:
recommend_properties('DLF The Arbour')

,PropertyName,SimilarityScore
64,Ace Palm Floors,"(63, 0.45293820624419556)"
217,Yashika 104,"(216, 0.41996063229267827)"
93,JMS The Nation,"(92, 0.4166584649363288)"
154,India Rashtra,"(153, 0.39895423468019414)"
0,Smartworld One DXP,"(0, 0.388850461994329)"


In [23]:
import pandas as pd
import json

# Load the dataset
df_appartments = pd.read_csv('apartments.csv', encoding='latin1').drop(22)

# Function to parse and extract the required features from the PriceDetails column
def refined_parse_modified_v2(detail_str):
    try:
        details = json.loads(detail_str.replace("'", "\""))
    except:
        return {}
        
    extracted = {}

    for bhk, detail in details.items():
        # Extract building type
        extracted[f'building type_{bhk}'] = detail.get('building type')

        # Parsing area details
        area = detail.get('area', '')
        area_parts = area.split('-')
        if len(area_parts) == 1:
            try:
                value = float(area_parts[0].replace(',', '').replace(' sq.ft', '').strip())
                extracted[f'area low{bhk}'] = value
                extracted[f'area high{bhk}'] = value
            except:
                extracted[f'area low{bhk}'] = None
                extracted[f'area high{bhk}'] = None

        elif len(area_parts) == 2:
            try:
                extracted[f'area low{bhk}'] = float(area_parts[0].replace(',', '').replace(' sq.ft', '').strip())
                extracted[f'area high{bhk}'] = float(area_parts[1].replace(',', '').replace(' sq.ft', '').strip())
            except:
                extracted[f'area low{bhk}'] = None
                extracted[f'area high{bhk}'] = None
            
        # Parsing price details
        price_range = detail.get('price-range', '')
        price_parts = price_range.split('-')
        if len(price_parts) == 2:
            try:
                extracted[f'price low{bhk}'] = float(price_parts[0].replace('₹', '').replace('cr', '').replace('L', '').strip())
                extracted[f'price high{bhk}'] = float(price_parts[1].replace('₹', '').replace('cr', '').replace('L', '').strip())
                if 'L' in price_parts[0]:
                    extracted[f'price low{bhk}'] /= 100
                if 'L' in price_parts[1]:
                    extracted[f'price high{bhk}'] /= 100
            
            except:
                extracted[f'price low{bhk}'] = None
                extracted[f'price high{bhk}'] = None
    return extracted

# Apply the refined parsing and generate new DataFrame structure
data_refined = []

for _, row in df_appartments.iterrows():
    features = refined_parse_modified_v2(row['PriceDetails'])

    # Construct a new row for the transformed dataframe
    new_row = {
        'PropertyName': row['PropertyName']}
    
    # Populate the new row with extracted features
    for config in ['1 BHK', '2 BHK', '3 BHK', '4 BHK', '5 BHK', '6 BHK', '1 RK', 'Land']:
        new_row[f'building type_{config}'] = features.get(f'building type_{config}')
        new_row[f'area low{config}'] = features.get(f'area low{config}')
        new_row[f'area high{config}'] = features.get(f'area high{config}')
        new_row[f'price low{config}'] = features.get(f'price low{config}')
        new_row[f'price high{config}'] = features.get(f'price high{config}')
    data_refined.append(new_row)

df_final_refined_v2 = pd.DataFrame(data_refined).set_index('PropertyName')

In [24]:
df_final_refined_v2['building type_Land'] = df_final_refined_v2['building type_Land'].replace({'':'Land'})

In [25]:
df['PriceDetails'][10]

"{'2 BHK': {'building_type': 'Independent Floor', 'area_type': 'Carpet Area', 'area': '1,055 sq.ft.', 'price-range': '? 1.05 - 1.5 Cr'}, '3 BHK': {'building_type': 'Independent Floor', 'area_type': 'Carpet Area', 'area': '1,325 - 1,525 sq.ft.', 'price-range': '? 1.35 - 1.84 Cr'}}"

In [26]:
df_final_refined_v2

,building type_1 BHK,area low1 BHK,area high1 BHK,price low1 BHK,price high1 BHK,building type_2 BHK,area low2 BHK,area high2 BHK,price low2 BHK,price high2 BHK,...,building type_1 RK,area low1 RK,area high1 RK,price low1 RK,price high1 RK,building type_Land,area lowLand,area highLand,price lowLand,price highLand
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,None,NaN,NaN,None,None,None,1370.0,1370.0,None,None,...,None,NaN,NaN,None,None,None,NaN,NaN,None,None
M3M Crown,None,NaN,NaN,None,None,None,NaN,NaN,None,None,...,None,NaN,NaN,None,None,None,NaN,NaN,None,None
Adani Brahma Samsara Vilasa,None,NaN,NaN,None,None,None,NaN,NaN,None,None,...,None,NaN,NaN,None,None,None,500.0,4329.0,None,None
Sobha City,None,NaN,NaN,None,None,None,1381.0,1692.0,None,None,...,None,NaN,NaN,None,None,None,NaN,NaN,None,None
Signature Global City 93,None,NaN,NaN,None,None,None,981.0,1118.0,None,None,...,None,NaN,NaN,None,None,None,NaN,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,None,NaN,NaN,None,None,None,964.0,964.0,None,None,...,None,NaN,NaN,None,None,None,NaN,NaN,None,None
Pyramid Urban Homes 2,None,335.0,398.0,None,None,None,500.0,625.0,None,None,...,None,NaN,NaN,None,None,None,NaN,NaN,None,None
Satya The Hermitage,None,NaN,NaN,None,None,None,1450.0,1450.0,None,None,...,None,NaN,NaN,None,None,None,NaN,NaN,None,None


In [27]:
categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()

In [28]:
categorical_columns

['building type_1 BHK',
 'price low1 BHK',
 'price high1 BHK',
 'building type_2 BHK',
 'price low2 BHK',
 'price high2 BHK',
 'building type_3 BHK',
 'price low3 BHK',
 'price high3 BHK',
 'building type_4 BHK',
 'price low4 BHK',
 'price high4 BHK',
 'building type_5 BHK',
 'price low5 BHK',
 'price high5 BHK',
 'building type_6 BHK',
 'price low6 BHK',
 'price high6 BHK',
 'building type_1 RK',
 'price low1 RK',
 'price high1 RK',
 'building type_Land',
 'price lowLand',
 'price highLand']

In [29]:
ohe_df = pd.get_dummies(df_final_refined_v2, columns=categorical_columns, drop_first=True)

In [30]:
ohe_df.fillna(0, inplace=True)

In [31]:
ohe_df

,area low1 BHK,area high1 BHK,area low2 BHK,area high2 BHK,area low3 BHK,area high3 BHK,area low4 BHK,area high4 BHK,area low5 BHK,area high5 BHK,area low6 BHK,area high6 BHK,area low1 RK,area high1 RK,area lowLand,area highLand
PropertyName,,,,,,,,,,,,,,,,
Smartworld One DXP,0.0,0.0,1370.0,1370.0,1850.0,2050.0,2600.0,2600.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
M3M Crown,0.0,0.0,0.0,0.0,1605.0,2170.0,2248.0,2670.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Adani Brahma Samsara Vilasa,0.0,0.0,0.0,0.0,1800.0,3150.0,2750.0,4500.0,0.0,0.0,0.0,0.0,0.0,0.0,500.0,4329.0
Sobha City,0.0,0.0,1381.0,1692.0,1711.0,2343.0,2423.0,2963.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Signature Global City 93,0.0,0.0,981.0,1118.0,1235.0,1530.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,0.0,0.0,964.0,964.0,1127.0,1127.0,1562.0,1562.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Pyramid Urban Homes 2,335.0,398.0,500.0,625.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Satya The Hermitage,0.0,0.0,1450.0,1450.0,1991.0,1991.0,2639.0,4711.0,4731.0,4731.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
ohe_df_normalized = pd.DataFrame(scaler.fit_transform(ohe_df), columns=ohe_df.columns, index=ohe_df.index)

In [33]:
ohe_df_normalized.head()

,area low1 BHK,area high1 BHK,area low2 BHK,area high2 BHK,area low3 BHK,area high3 BHK,area low4 BHK,area high4 BHK,area low5 BHK,area high5 BHK,area low6 BHK,area high6 BHK,area low1 RK,area high1 RK,area lowLand,area highLand
PropertyName,,,,,,,,,,,,,,,,
Smartworld One DXP,-0.17632,-0.175974,1.322202,1.120892,0.691042,0.489481,0.657860,0.273965,-0.422661,-0.417419,-0.125582,-0.118934,-0.09051,-0.082059,-0.404036,-0.342467
M3M Crown,-0.17632,-0.175974,-0.766647,-0.771007,0.431655,0.589666,0.441060,0.304853,-0.422661,-0.417419,-0.125582,-0.118934,-0.09051,-0.082059,-0.404036,-0.342467
Adani Brahma Samsara Vilasa,-0.17632,-0.175974,-0.766647,-0.771007,0.638106,1.407844,0.750246,1.112368,-0.422661,-0.417419,-0.125582,-0.118934,-0.09051,-0.082059,0.427231,2.005303
Sobha City,-0.17632,-0.175974,1.338974,1.565557,0.543880,0.734100,0.548844,0.434144,-0.422661,-0.417419,-0.125582,-0.118934,-0.09051,-0.082059,-0.404036,-0.342467
Signature Global City 93,-0.17632,-0.175974,0.729091,0.772893,0.039926,0.055346,-0.943503,-0.873325,-0.422661,-0.417419,-0.125582,-0.118934,-0.09051,-0.082059,-0.404036,-0.342467


In [34]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix
cosine_sim2 = cosine_similarity(ohe_df_normalized)

In [35]:
cosine_sim2.shape

(246, 246)

In [36]:
def recommend_properties_with_scores(property_name, top_n=247):

    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim2[ohe_df_normalized.index.get_loc(property_name)]))

    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]

    # Retrieve the names of the top propeties using the indices
    top_properties = ohe_df_normalized.index[top_indices].tolist()

    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })

    return recommendations_df

#  Test the recommender function using a property name
recommend_properties_with_scores('International City by SOBHA Phase 2')

,PropertyName,SimilarityScore
0,DLF The Summit,0.989982
1,DLF The Pinnacle,0.984996
2,Mapsko The Icon 79,0.983675
3,DLF The Arbour,0.982042
4,BPTP Mansions Park Prime,0.957106
...,...,...
240,M3M Skycity,-0.672436
241,M3M Heights,-0.674330
242,Shree Vardhman Victoria,-0.683091
243,Godrej Aria,-0.688193


In [37]:
df[['PropertyName', 'LocationAdvantages']]['LocationAdvantages'][0]

"{'Bajghera Road': '800 Meter', 'Palam Vihar Halt': '2.5 KM', 'DPSG Palam Vihar': '3.1 KM', 'Park Hospital': '3.1 KM', 'Gurgaon Railway Station': '4.9 KM', 'The NorthCap University': '5.4 KM', 'Dwarka Expy': '1.2 KM', 'Hyatt Place Gurgaon Udyog Vihar': '7.7 KM', 'Dwarka Sector 21, Metro Station': '7.2 KM', 'Pacific D21 Mall': '7.4 KM', 'Indira Gandhi International Airport': '14.7 KM', 'Hamoni Golf Camp': '6.2 KM', 'Fun N Food Waterpark': '8.8 KM', 'Accenture DDC5': '9 KM'}"

In [38]:
def distance_to_meters(distance_str):
    try:
        if 'km' in distance_str or 'KM' in distance_str:
            return float(distance_str.split()[0]) * 1000
        elif 'Meter' in distance_str or 'meter' in distance_str:
            return float(distance_str.split()[0])
        else:
            return None
    except:
        return None

In [39]:
import ast

# Extract distances for each location
location_matrix = {}
for index, row in df.iterrows():
    distances = {}
    for location, distance in ast.literal_eval(row['LocationAdvantages']).items():
        distances[location] = distance_to_meters(distance)
    location_matrix[index] = distances

# Convert the dictonary to a dataframe
location_df = pd.DataFrame.from_dict(location_matrix, orient='index')

# Display the first few rows
location_df.head()

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,...,MCC Cricket Ground Dhankot,The Shri Ram School Aravali,Taj City Centre Gurugram,Minda Industries Corporate Office,"Rampura Flyover, Naurangpur Rd",Manesar toll plaza - Kherki Daula,"Imt Manesar, Gurugram",Holiday Inn,Sector 84 Road,Skyview Corporate Park
0,800.0,2500.0,3100.0,3100.0,4900.0,5400.0,1200.0,7700.0,7200.0,7400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,550.0,NaN,NaN,NaN,NaN,6700.0,3800.0,NaN,NaN,7500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37,5300.0,NaN,NaN,NaN,2500.0,8800.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,1500.0,NaN,NaN,NaN,6500.0,6700.0,5100.0,NaN,NaN,8200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,5500.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
location_df.columns[10:50]

Index(['Indira Gandhi International Airport', 'Hamoni Golf Camp',
       'Fun N Food Waterpark', 'Accenture DDC5', 'DPSG Palam Vihar Gurugram',
       'Park Hospital, Palam Vihar', 'Palam Vihar Halt Railway Station',
       'Dwarka Sector 21 Metro Station', 'Dwarka Expressway',
       'Fun N Food Water Park', 'Tau DeviLal Sports Complex', 'Hyatt Place',
       'Altrade Business Centre', 'AIPL Business Club Sector 62',
       'Heritage Xperiential Learning School', 'CK Birla Hospital',
       'Paras Trinity Mall Sector 63', 'Rapid Metro Station Sector 56',
       'De Adventure Park', 'Golf Course Ext Rd',
       'DoubleTree by Hilton Hotel Gurgaon',
       'KIIT College of Engineering Sohna Road', 'Mehrauli-Gurgaon Road',
       'Nirvana Rd', 'TERI Golf Course', 'The Shikshiyan School', 'WTC Plaza',
       'Luxus Haritma Resort', 'BSF Golf Course', 'Rions Hospital', 'Gurgaon',
       'Dwarka Sector 21', 'Nehru Stadium', 'Fun N Food WaterPark',
       'IGI Airport', 'Vasant Kunj', 'Prana

In [41]:
location_df.index = df.PropertyName

In [42]:
location_df.head()

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,...,MCC Cricket Ground Dhankot,The Shri Ram School Aravali,Taj City Centre Gurugram,Minda Industries Corporate Office,"Rampura Flyover, Naurangpur Rd",Manesar toll plaza - Kherki Daula,"Imt Manesar, Gurugram",Holiday Inn,Sector 84 Road,Skyview Corporate Park
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,800.0,2500.0,3100.0,3100.0,4900.0,5400.0,1200.0,7700.0,7200.0,7400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
M3M Crown,550.0,NaN,NaN,NaN,NaN,6700.0,3800.0,NaN,NaN,7500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Adani Brahma Samsara Vilasa,5300.0,NaN,NaN,NaN,2500.0,8800.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sobha City,1500.0,NaN,NaN,NaN,6500.0,6700.0,5100.0,NaN,NaN,8200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Signature Global City 93,NaN,NaN,NaN,5500.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
location_df.fillna(54000,inplace=True)

In [44]:
location_df

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,...,MCC Cricket Ground Dhankot,The Shri Ram School Aravali,Taj City Centre Gurugram,Minda Industries Corporate Office,"Rampura Flyover, Naurangpur Rd",Manesar toll plaza - Kherki Daula,"Imt Manesar, Gurugram",Holiday Inn,Sector 84 Road,Skyview Corporate Park
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,800.0,2500.0,3100.0,3100.0,4900.0,5400.0,1200.0,7700.0,7200.0,7400.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
M3M Crown,550.0,54000.0,54000.0,54000.0,54000.0,6700.0,3800.0,54000.0,54000.0,7500.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
Adani Brahma Samsara Vilasa,5300.0,54000.0,54000.0,54000.0,2500.0,8800.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
Sobha City,1500.0,54000.0,54000.0,54000.0,6500.0,6700.0,5100.0,54000.0,54000.0,8200.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
Signature Global City 93,54000.0,54000.0,54000.0,5500.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
Pyramid Urban Homes 2,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
Satya The Hermitage,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0


In [45]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
location_df_normalized = pd.DataFrame(scaler.fit_transform(location_df), columns=location_df.columns, index=location_df.index)

In [46]:
location_df_normalized

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,...,MCC Cricket Ground Dhankot,The Shri Ram School Aravali,Taj City Centre Gurugram,Minda Industries Corporate Office,"Rampura Flyover, Naurangpur Rd",Manesar toll plaza - Kherki Daula,"Imt Manesar, Gurugram",Holiday Inn,Sector 84 Road,Skyview Corporate Park
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,-7.960979,-15.652476,-15.652476,-3.468082,-4.372967,-4.275539,-4.110980,-10.231739,-15.652476,-7.133231,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888
M3M Crown,-7.998993,0.063888,0.063888,0.304647,0.260429,-4.154617,-3.896001,0.090308,0.063888,-7.117615,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888
Adani Brahma Samsara Vilasa,-7.276720,0.063888,0.063888,0.304647,-4.599447,-3.959281,0.254741,0.090308,0.063888,0.143975,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888
Sobha City,-7.854539,0.063888,0.063888,0.304647,-4.221981,-4.154617,-3.788512,0.090308,0.063888,-7.008301,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888
Signature Global City 93,0.128476,0.063888,0.063888,-3.290193,0.260429,0.245097,0.254741,0.090308,0.063888,0.143975,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,0.128476,0.063888,0.063888,0.304647,0.260429,0.245097,0.254741,0.090308,0.063888,0.143975,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888
Pyramid Urban Homes 2,0.128476,0.063888,0.063888,0.304647,0.260429,0.245097,0.254741,0.090308,0.063888,0.143975,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888
Satya The Hermitage,0.128476,0.063888,0.063888,0.304647,0.260429,0.245097,0.254741,0.090308,0.063888,0.143975,...,0.0,0.0,0.0,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888,0.063888


In [47]:
cosine_sim3 = cosine_similarity(location_df_normalized)

In [48]:
cosine_sim3.shape

(246, 246)

In [49]:
def recommend_properties_with_scores(property_name, top_n=247):

    cosine_sim_matrix = 30*cosine_sim1 + 20*cosine_sim2 + 8*cosine_sim3

    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))

    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]

    # Retrieve the names of the top propeties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()

    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })

    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('DLF The Arbour')

,PropertyName,SimilarityScore
0,SS Linden Floors,28.490531
1,DLF The Summit,25.840527
2,Tulip Purple,25.175006
3,BPTP Mansions Park Prime,24.680205
4,DLF The Pinnacle,23.256951
...,...,...
240,M3M Skycity,-12.262032
241,Pareena Mi Casa,-12.360652
242,Godrej 101,-12.377959
243,Unitech Fresco,-12.846392


In [50]:
(3*cosine_sim3 + 5*cosine_sim2 + 6*cosine_sim1).shape

(246, 246)

In [51]:
import pickle

pickle.dump(location_df, open('location_distance.pkl','wb'))

In [54]:
pickle.dump(cosine_sim1, open('cosine_sim1.pkl','wb'))
pickle.dump(cosine_sim2, open('cosine_sim2.pkl','wb'))
pickle.dump(cosine_sim3, open('cosine_sim3.pkl','wb'))